<a href="https://colab.research.google.com/github/rodrigorissettoterra/SENAI_Concepcao_e_Design_de_ML/blob/main/Dados_Hepatite_Ativ_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Carregamento dos dados

In [ ]:
import pandas as pd

url = "https://www.openml.org/data/get_csv/55/dataset_55_hepatitis.arff"
df = pd.read_csv(url)

df.head()

,AGE,SEX,STEROID,ANTIVIRALS,FATIGUE,MALAISE,ANOREXIA,LIVER_BIG,LIVER_FIRM,SPLEEN_PALPABLE,SPIDERS,ASCITES,VARICES,BILIRUBIN,ALK_PHOSPHATE,SGOT,ALBUMIN,PROTIME,HISTOLOGY,Class
0,30,male,no,no,no,no,no,no,no,no,no,no,no,1,85,18,4,?,no,LIVE
1,50,female,no,no,yes,no,no,no,no,no,no,no,no,0.9,135,42,3.5,?,no,LIVE
2,78,female,yes,no,yes,no,no,yes,no,no,no,no,no,0.7,96,32,4,?,no,LIVE
3,31,female,?,yes,no,no,no,yes,no,no,no,no,no,0.7,46,52,4,80,no,LIVE
4,34,female,yes,no,no,no,no,yes,no,no,no,no,no,1,?,200,4,?,no,LIVE


Exploração inicial

In [ ]:
df.info()
df.describe()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 155 entries, 0 to 154
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   AGE              155 non-null    int64 
 1   SEX              155 non-null    object
 2   STEROID          155 non-null    object
 3   ANTIVIRALS       155 non-null    object
 4   FATIGUE          155 non-null    object
 5   MALAISE          155 non-null    object
 6   ANOREXIA         155 non-null    object
 7   LIVER_BIG        155 non-null    object
 8   LIVER_FIRM       155 non-null    object
 9   SPLEEN_PALPABLE  155 non-null    object
 10  SPIDERS          155 non-null    object
 11  ASCITES          155 non-null    object
 12  VARICES          155 non-null    object
 13  BILIRUBIN        155 non-null    object
 14  ALK_PHOSPHATE    155 non-null    object
 15  SGOT             155 non-null    object
 16  ALBUMIN          155 non-null    object
 17  PROTIME          155 non-null    ob

,0
AGE,0
SEX,0
STEROID,0
ANTIVIRALS,0
FATIGUE,0
MALAISE,0
ANOREXIA,0
LIVER_BIG,0
LIVER_FIRM,0
SPLEEN_PALPABLE,0


Tratamento de tipos de dados

In [ ]:
categorical_cols = [
    'SEX', 'STEROID', 'ANTIVIRALS', 'FATIGUE', 'MALAISE', 'ANOREXIA',
    'LIVER_BIG', 'LIVER_FIRM', 'SPLEEN_PALPABLE', 'SPIDERS',
    'ASCITES', 'VARICES', 'HISTOLOGY', 'Class'
]

for col in categorical_cols:
    df[col] = df[col].astype('category')

Converter valores inválidos ("?") para NaN

In [ ]:
df.replace('?', pd.NA, inplace=True)

/tmp/ipykernel_3429/3043903598.py:1: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df.replace('?', pd.NA, inplace=True)


Converter colunas numéricas (que estão como object)

In [ ]:
num_cols = ['BILIRUBIN', 'ALK_PHOSPHATE', 'SGOT', 'ALBUMIN', 'PROTIME']

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

Verificar valores faltantes novamente

In [ ]:
df.isnull().sum()

,0
AGE,0
SEX,0
STEROID,1
ANTIVIRALS,0
FATIGUE,1
MALAISE,1
ANOREXIA,1
LIVER_BIG,10
LIVER_FIRM,11
SPLEEN_PALPABLE,5


Tratar valores faltantes

In [ ]:
# Numéricos
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Categóricos
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

Remover duplicados

In [ ]:
df = df.drop_duplicates()

Verificar outliers com IQR

In [ ]:
Q1 = df[num_cols].quantile(0.25)
Q3 = df[num_cols].quantile(0.75)
IQR = Q3 - Q1

filtro = ~((df[num_cols] < (Q1 - 1.5 * IQR)) | (df[num_cols] > (Q3 + 1.5 * IQR))).any(axis=1)
df = df[filtro]

Padronizar dados numéricos

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

Transformar categóricas em numéricas

In [ ]:
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

Conferir resultado

In [ ]:
df.info()
df.head()
df.isnull().sum().sum()

<class 'pandas.core.frame.DataFrame'>
Index: 85 entries, 0 to 153
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   AGE                  85 non-null     int64  
 1   BILIRUBIN            85 non-null     float64
 2   ALK_PHOSPHATE        85 non-null     float64
 3   SGOT                 85 non-null     float64
 4   ALBUMIN              85 non-null     float64
 5   PROTIME              85 non-null     float64
 6   SEX_male             85 non-null     bool   
 7   STEROID_yes          85 non-null     bool   
 8   ANTIVIRALS_yes       85 non-null     bool   
 9   FATIGUE_yes          85 non-null     bool   
 10  MALAISE_yes          85 non-null     bool   
 11  ANOREXIA_yes         85 non-null     bool   
 12  LIVER_BIG_yes        85 non-null     bool   
 13  LIVER_FIRM_yes       85 non-null     bool   
 14  SPLEEN_PALPABLE_yes  85 non-null     bool   
 15  SPIDERS_yes          85 non-null     bool   
 

np.int64(0)